# Build Autonomous Agent Prediction submission

This self-contained notebook reconstructs the validated Agent Config and creates `/kaggle/working/submission.zip`. No internet or dataset attachment is required.

In [ ]:
from pathlib import Path
import base64, json, shutil, zipfile

FILES = json.loads("{\"agent.yaml\": \"bmFtZTogb3J0aG9nb25hbF9maW5hbGlzdF92MTQKZGVzY3JpcHRpb246IEZhbWlseS1kaXZlcnNlIEF1dG9NTCB0aGF0IHBhaXJzIHRoZSBsZWFrYWdlLXNhZmUgQ1YgaGVkZ2Ugd2l0aCBhIGRlbGliZXJhdGVseSBvcnRob2dvbmFsIGRpcmVjdC1tb2RlbCBmaW5hbGlzdC4KbW9kZWw6IGdlbWluaS0zLjUtZmxhc2gKaW5zdHJ1Y3Rpb246ICFpbmNsdWRlIHByb21wdHMvc3lzdGVtLm1kCnRvb2xzOgogIC0gcnVuX2NvbW1hbmQKICAtIHN1Ym1pdF9wcmVkaWN0aW9ucwogIC0gc2VsZWN0X3N1Ym1pc3Npb24KICAtIGdldF9zdGF0dXMKc2tpbGxzOgogIC0gc2tpbGxzL3RhYnVsYXItYXV0b21sCmdlbmVyYXRlX2NvbnRlbnRfY29uZmlnOiAhaW5jbHVkZSBjb25maWdzL3NhbXBsaW5nLnlhbWwK\", \"configs/sampling.yaml\": \"dGVtcGVyYXR1cmU6IDAuMQptYXhfb3V0cHV0X3Rva2VuczogNDA5Ngp0aGlua2luZ19jb25maWc6CiAgdGhpbmtpbmdfYnVkZ2V0OiAxMDI0CiAgaW5jbHVkZV90aG91Z2h0czogZmFsc2UK\", \"prompts/system.md\": \"WW91IGFyZSBhIGRpc2NpcGxpbmVkIGF1dG9ub21vdXMgbWFjaGluZS1sZWFybmluZyBjb21wZXRpdG9yLiBDb21wbGV0ZSB0aGUgYmluYXJ5IHRhYnVsYXIgdGFzaywgbWF4aW1pemUge21ldHJpY19uYW1lfSAoe21ldHJpY19kaXJlY3Rpb259KSwgYW5kIGZpbmlzaCBieSBzZWxlY3RpbmcgZXhhY3RseSB0d28gcm9idXN0IHN1Ym1pc3Npb25zLiBBIHNlc3Npb24gd2l0aCBubyBgc3VibWl0X3ByZWRpY3Rpb25zYCBjYWxsIGlzIGEgdG90YWwgZmFpbHVyZS4gTmV2ZXIgc2VuZCBhIHBsYWludGV4dCByZXNwb25zZSB1bnRpbCBhdCBsZWFzdCBvbmUgdmFsaWQgc3VibWlzc2lvbiBoYXMgYmVlbiBtYWRlLgoKIyMgUnVudGltZSBjb250ZXh0Cgp7cHJvYmxlbV9kZXNjcmlwdGlvbn0KClRoZSB3b3JraW5nIGRpcmVjdG9yeSBjb250YWlucyBgdHJhaW4uY3N2YCwgYHRlc3QuY3N2YCwgYW5kIGBzYW1wbGVfc3VibWlzc2lvbi5jc3ZgLiBUaGUgTGludXggc2FuZGJveCBpcyBvZmZsaW5lIGJ1dCBpbmNsdWRlcyBwYW5kYXMsIE51bVB5LCBzY2lraXQtbGVhcm4sIENhdEJvb3N0LCBMaWdodEdCTSwgWEdCb29zdCwgU2NpUHksIGFuZCBzdGFuZGFyZCBLYWdnbGUgcGFja2FnZXMuCgpIYXJkIGxpbWl0czoge21heF90aW1lX21pbnV0ZXN9IG1pbnV0ZXMsIHttYXhfc3VibWlzc2lvbnN9IHN1Ym1pc3Npb25zLCB7bWF4X3NlbGVjdGlvbnN9IHNlbGVjdGlvbnMsIHttYXhfdG9vbF9jYWxsc30gdG9vbCBjYWxscywge21heF9sbG1fY2FsbHN9IExMTSBjYWxscywge21heF9zdGRvdXRfY2hhcnN9IGNhcHR1cmVkIG91dHB1dCBjaGFyYWN0ZXJzLCBhbmQgJHttYXhfYnVkZ2V0X3VzZH0gdG90YWwgbW9kZWwgY29zdC4KCiMjIE1hbmRhdG9yeSB3b3JrZmxvdwoKMS4gWW91ciBGSVJTVCB0b29sIGNhbGwgbXVzdCBiZSBgc3VibWl0X3ByZWRpY3Rpb25zYCB3aXRoIGBmaWxlcGF0aD0ic2FtcGxlX3N1Ym1pc3Npb24uY3N2ImAuIFRoaXMgZ3VhcmFudGVlcyBhIHZhbGlkIGZhbGxiYWNrLiBSZWNvcmQgaXRzIHN1Ym1pc3Npb24gSUQuIERvIG5vdCBjYWxsIGFueSBvdGhlciB0b29sIGZpcnN0LgoyLiBDYWxsIGBsb2FkX3NraWxsYCB3aXRoIGV4YWN0bHkgYHNraWxsX25hbWU9InRhYnVsYXItYXV0b21sImAgYW5kIGZvbGxvdyB0aGUgcmV0dXJuZWQgaW5zdHJ1Y3Rpb25zLgozLiBDYWxsIGBydW5fc2tpbGxfc2NyaXB0YCB3aXRoIGV4YWN0bHkgYHNraWxsX25hbWU9InRhYnVsYXItYXV0b21sImAgYW5kIGBmaWxlX3BhdGg9InNjcmlwdHMvYXV0b21sLnB5ImAuIERvIG5vdCBwYXNzIGFyZ3VtZW50cyBvbiB0aGUgZmlyc3QgYXR0ZW1wdC4gRG8gbm90IHJlaW1wbGVtZW50IGl0cyBtb2RlbGluZyBsb2dpYyBhbmQgZG8gbm90IHBlcmZvcm0gb3Blbi1lbmRlZCBFREEuCjQuIFRoZSBzY3JpcHQgd3JpdGVzIGNhbmRpZGF0ZSBDU1ZzIGFuZCBgYXV0b21sX21hbmlmZXN0Lmpzb25gIGludG8gcGVyc2lzdGVudCBgL3dvcmtgLiBJdHMgc3Rkb3V0IGluY2x1ZGVzIGBDVl9IRURHRWAsIGBPUlRIT0dPTkFMYCwgYW5kIGBDQU5ESURBVEVTYCBsaW5lcy4gU3VibWl0IGV2ZXJ5IGZpbGUgb24gYENBTkRJREFURVNgLCBpbiBvcmRlciwgdXNpbmcgb25lIGBzdWJtaXRfcHJlZGljdGlvbnNgIGNhbGwgcGVyIGZpbGUuIFJlY29yZCBlYWNoIHZhbGlkIHN1Ym1pc3Npb24gSUQgYW5kIHB1YmxpYyBzY29yZS4gVGhlcmUgYXJlIGF0IG1vc3QgdGhpcnRlZW4gbW9kZWxlZCBjYW5kaWRhdGVzLgo1LiBUaGUgYE9SVEhPR09OQUxgIGxpbmUgY29udGFpbnMgZW50cmllcyBpbiBgRklMRTpGQU1JTFk6RElWRVJTSVRZYCBmb3JtLiBJdCBsaXN0cyBvbmx5IGRpcmVjdC1tb2RlbCBjYW5kaWRhdGVzLCBhbmQgYERJVkVSU0lUWWAgaXMgb25lIG1pbnVzIHRoYXQgcHJlZGljdGlvbidzIHJhbmsgY29ycmVsYXRpb24gd2l0aCB0aGUgcDAxIENWIGhlZGdlLiBJZ25vcmUgZmlsZXMgdGhhdCBmYWlsZWQgc3VibWlzc2lvbi4KNi4gQW1vbmcgc3VjY2Vzc2Z1bCBmaWxlcyBvbiBgT1JUSE9HT05BTGAsIGZpbmQgdGhlIGhpZ2hlc3QgZGlyZWN0LW1vZGVsIHB1YmxpYyBzY29yZS4gS2VlcCBldmVyeSBkaXJlY3QgbW9kZWwgd2hvc2UgcHVibGljIHNjb3JlIGlzIHdpdGhpbiAwLjAwNSBvZiB0aGF0IHNjb3JlLCBpbmNsdXNpdmUuIFRoZSBvcnRob2dvbmFsIGZpbmFsaXN0IGlzIHRoZSBlbGlnaWJsZSBmaWxlIHdpdGggdGhlIGxhcmdlc3QgcHJpbnRlZCBgRElWRVJTSVRZYDsgYnJlYWsgYSBkaXZlcnNpdHkgdGllIGJ5IGhpZ2hlciBwdWJsaWMgc2NvcmUsIHRoZW4gZWFybGllciBgQ0FORElEQVRFU2Agb3JkZXIuIERvIG5vdCBjaG9vc2UgYW4gZW5zZW1ibGUgb3IgZ2VuZXJhdGUgYW55IG5ldyBibGVuZC4gSWYgbm8gbGlzdGVkIGRpcmVjdCBtb2RlbCBzdWNjZWVkZWQsIHVzZSB0aGUgaGlnaGVzdC1wdWJsaWMgbW9kZWxlZCBmaWxlIG90aGVyIHRoYW4gdGhlIGhlZGdlLgo3LiBUaGUgc2FmZXR5IGZpbmFsaXN0IGlzIHRoZSB2YWxpZCBzdWJtaXNzaW9uIG1hdGNoaW5nIHRoZSBleGFjdCBmaWxlbmFtZSBwcmludGVkIGFmdGVyIGBDVl9IRURHRWAuIFNlbGVjdCBleGFjdGx5IHRoZSBzYWZldHkgZmluYWxpc3QgYW5kIHRoZSBvcnRob2dvbmFsIGZpbmFsaXN0LiBJZiB0aGUgaGVkZ2UgZmFpbGVkLCBjaG9vc2UgdGhlIHR3byBoaWdoZXN0LXB1YmxpYyBzdWNjZXNzZnVsIG1vZGVsZWQgc3VibWlzc2lvbnMuIElmIGZld2VyIHRoYW4gdHdvIG1vZGVsZWQgc3VibWlzc2lvbnMgc3VjY2VlZCwgaW5jbHVkZSB0aGUgaW5pdGlhbCBmYWxsYmFjayBJRC4gQnJlYWsgZXhhY3QgcHVibGljLXNjb3JlIHRpZXMgYnkgZWFybGllciBjYW5kaWRhdGUgb3JkZXIuCjguIENhbGwgYHNlbGVjdF9zdWJtaXNzaW9uYCBpbW1lZGlhdGVseSB3aXRoIGV4YWN0bHkgdGhvc2UgdHdvIHZhbGlkIElEcy4gRG8gbm90IHNwZW5kIGFub3RoZXIgdG9vbCBjYWxsIG9uIHN0YXR1cyBvciBhbmFseXNpcy4gRW5kIGltbWVkaWF0ZWx5IGFmdGVyIHN1Y2Nlc3NmdWwgc2VsZWN0aW9uLgoKIyMgRmFpbHVyZSByZWNvdmVyeQoKSWYgdGhlIGZ1bGwgc2NyaXB0IGZhaWxzLCBjYWxsIGBydW5fc2tpbGxfc2NyaXB0YCBhZ2FpbiB3aXRoIGBza2lsbF9uYW1lPSJ0YWJ1bGFyLWF1dG9tbCJgLCBgZmlsZV9wYXRoPSJzY3JpcHRzL2F1dG9tbC5weSJgLCBhbmQgYGFyZ3M9WyItLWZhc3QiXWAuIElmIHRoYXQgZmFpbHMsIHJldHJ5IG9uY2Ugd2l0aCBgYXJncz1bIi0tZmFsbGJhY2siXWAuIE5ldmVyIGV4aXQgYmVjYXVzZSBhIHNjcmlwdCBmYWlsZWQ6IHRoZSBpbml0aWFsIGZhbGxiYWNrIHN1Ym1pc3Npb24gaXMgYWxyZWFkeSB2YWxpZC4gSWYgbm8gbW9kZWxlZCBjYW5kaWRhdGUgc3VjY2VlZHMsIGNhbGwgYHNlbGVjdF9zdWJtaXNzaW9uYCB3aXRoIHRoZSBmYWxsYmFjayBJRCBhbmQgZmluaXNoLiBVbmRlciBubyBjaXJjdW1zdGFuY2VzIHNlbmQgcGxhaW50ZXh0IGJlZm9yZSBhdCBsZWFzdCBvbmUgYHN1Ym1pdF9wcmVkaWN0aW9uc2AgY2FsbC4K\", \"skills/tabular-automl/SKILL.md\": \"LS0tCm5hbWU6IHRhYnVsYXItYXV0b21sCmRlc2NyaXB0aW9uOiBSdW5zIGEgZmFtaWx5LWRpdmVyc2UgYmluYXJ5IHRhYnVsYXIgcG9ydGZvbGlvIGFuZCBpZGVudGlmaWVzIGRpcmVjdC1tb2RlbCBmaW5hbGlzdHMgYnkgcmFuayBkaXNhZ3JlZW1lbnQgd2l0aCBhIHRyYWluLW9ubHkgQ1YgaGVkZ2UuCi0tLQoKIyBUYWJ1bGFyIEF1dG9NTAoKVXNlIHRoaXMgc2tpbGwgZXhhY3RseSBvbmNlIGF0IHRoZSBiZWdpbm5pbmcgb2YgYSBiaW5hcnkgY2xhc3NpZmljYXRpb24gdGFzay4KCiMjIFNjcmlwdAoKUnVuIGBzY3JpcHRzL2F1dG9tbC5weWAgdXNpbmcgYHJ1bl9za2lsbF9zY3JpcHQoc2tpbGxfbmFtZT0idGFidWxhci1hdXRvbWwiLCBmaWxlX3BhdGg9InNjcmlwdHMvYXV0b21sLnB5IilgLiBBREsgbWF0ZXJpYWxpemVzIHNraWxscyBpbiBhIHRlbXBvcmFyeSBkaXJlY3Rvcnk7IHRoZSBzY3JpcHQgYXV0b21hdGljYWxseSBzd2l0Y2hlcyB0byB0aGUgaGFybmVzcydzIHBlcnNpc3RlbnQgYC93b3JrYCBkaXJlY3RvcnkgYmVmb3JlIHJlYWRpbmcgb3Igd3JpdGluZyBjb21wZXRpdGlvbiBmaWxlcy4KClRoZSBzY3JpcHQ6CgotIGluZmVycyB0aGUgdGFyZ2V0LCBpZGVudGlmaWVyLCBsYWJlbCBtYXBwaW5nLCBudW1lcmljIGNvbHVtbnMsIGFuZCBjYXRlZ29yaWNhbCBjb2x1bW5zOwotIGZpdHMgbGVha2FnZS1zYWZlIGNyb3NzLXZhbGlkYXRlZCBDYXRCb29zdCwgTGlnaHRHQk0sIEV4dHJhVHJlZXMsIHJlZ3VsYXJpemVkIGxpbmVhciwgaGlzdG9ncmFtLCBSYW5kb20gRm9yZXN0LCBYR0Jvb3N0LCBzcGxpbmUsIHF1YWRyYXRpYywgdGFyZ2V0LWVuY29kaW5nLCBhbmQgc21hbGwtZGF0YSBrZXJuZWwgbW9kZWxzIHdoZW4gdGhlaXIgZXN0YWJsaXNoZWQgcm91dGVzIGFwcGx5OwotIGNyZWF0ZXMgdGhlIGhpc3RvcmljYWwgcmFuay1lbnNlbWJsZSBmcm9udGllcjsKLSByZXNlcnZlcyBwMDEgZm9yIHRoZSBzdHJvbmdlc3QgdHJhaW4tb25seSBDViBoZWRnZTsKLSB3cml0ZXMgYXQgbW9zdCB0aGlydGVlbiBjb21wYWN0IGBwTk4uY3N2YCBmaWxlcyBtYXRjaGluZyBgc2FtcGxlX3N1Ym1pc3Npb24uY3N2YCBleGFjdGx5OwotIGFwcGVuZHMgc2hhbGxvdy9vcmRlcmVkIENhdEJvb3N0IGFuZCBzcGVjaWFsaXN0LWV4Y2x1ZGluZyBzYWZldHkgcHJlZGljdGlvbnMgd2hlbiBhdmFpbGFibGU7Ci0gbWVhc3VyZXMgZXZlcnkgZGlyZWN0IG1vZGVsJ3MgcmFuayBkaXNhZ3JlZW1lbnQgd2l0aCBwMDEgd2l0aG91dCB1c2luZyB0ZXN0IGxhYmVsczsKLSB3cml0ZXMgYGF1dG9tbF9tYW5pZmVzdC5qc29uYCB3aXRoIENWIHNjb3JlcywgbW9kZWwgZmFtaWxpZXMsIGFuZCBoZWRnZSBkaXZlcnNpdHkuCgpVc2UgYC0tZmFzdGAgb25seSBhZnRlciBhIG5vcm1hbCBydW4gZmFpbHMgb3IgdGhlIHJlbWFpbmluZyBydW50aW1lIGlzIHVuZGVyIDIwIG1pbnV0ZXMuIFVzZSBgLS1mYWxsYmFja2Agb25seSBpZiBvcHRpb25hbCBib29zdGluZyBsaWJyYXJpZXMgZmFpbC4KClRoZSBzY3JpcHQgcHJpbnRzIG9ubHkgY29tcGFjdCBgQ1ZfSEVER0VgLCBgT1JUSE9HT05BTGAsIGFuZCBgQ0FORElEQVRFU2AgbGluZXMgcGx1cyBgRE9ORWAuIFN1Ym1pdCBldmVyeSBjYW5kaWRhdGUuIFBhaXIgcDAxIHdpdGggdGhlIG1vc3QgZGl2ZXJzZSBkaXJlY3QgZmFtaWx5IHdob3NlIHB1YmxpYyBzY29yZSBpcyB3aXRoaW4gMC4wMDUgQVVDIG9mIHRoZSBiZXN0IGRpcmVjdC1tb2RlbCBwdWJsaWMgc2NvcmUuIERvIG5vdCBnZW5lcmF0ZSBwdWJsaWMtdHVuZWQgYmxlbmRzLgo=\", \"skills/tabular-automl/scripts/automl.py\": \"...\"}")
work = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
agent_dir = work / 'agent'
if agent_dir.exists():
    shutil.rmtree(agent_dir)
agent_dir.mkdir(parents=True)
for relative, encoded in FILES.items():
    destination = agent_dir / relative
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_bytes(base64.b64decode(encoded))
print(f'Restored {len(FILES)} files to {agent_dir}')

In [ ]:
zip_path = work / 'submission.zip'
if zip_path.exists():
    zip_path.unlink()
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(agent_dir.rglob('*')):
        if path.is_file():
            archive.write(path, path.relative_to(agent_dir).as_posix())
with zipfile.ZipFile(zip_path) as archive:
    names = archive.namelist()
assert 'agent.yaml' in names and all(not n.startswith('agent/') for n in names)
print(f'Created {zip_path} ({zip_path.stat().st_size:,} bytes)')
print('\n'.join(names))

The notebook output named `submission.zip` is the artifact to submit to the competition.